In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from langchain_deepseek import ChatDeepSeek
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain.agents import create_agent
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool


C:\Users\ASUS TUF A15\AppData\Local\Temp\ipykernel_12776\935764103.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import GoogleSerperAPIWrapper


In [3]:
class FlowState(BaseModel):
    question:str=Field(description="user question")
    category: Literal['coding', 'google_search', 'weather'] = Field(default="google_search") # প্রশ্নের ধরন
    answer: str = Field(default="")

In [4]:
class QuestionCategory(BaseModel):
    category: Literal['coding', 'google_search', 'weather'] = Field(
        default="google_search",
        description="Question Category"
    )

In [5]:
llm = ChatDeepSeek(model="deepseek-chat")

In [6]:
search=GoogleSerperAPIWrapper(serper_api_key="37c565decdbf74c2ac8ab1e1404783160e4e093d")
@tool
def google_search_tool(query: str) -> str:
    """Search Google for recent information, facts, and news."""
    return search.run(query)

tools = [google_search_tool]

google_agent=create_react_agent(
    model=llm,
    tools=tools,
    prompt="You are a agent search any queastion in google"
    
)


C:\Users\ASUS TUF A15\AppData\Local\Temp\ipykernel_12776\1067715924.py:9: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  google_agent=create_react_agent(


In [7]:
def check_question_category(state: FlowState) -> FlowState:
    # LLM-কে বাধ্য করা হচ্ছে যাতে সে শুধুমাত্র QuestionCategory মডেলে আউটপুট দেয়
    st_llm = llm.with_structured_output(QuestionCategory)
    
    # প্রশ্ন পাঠিয়ে ক্যাটাগরি চাওয়া হচ্ছে
    res = st_llm.invoke(f"I want to know the category of my question, question is : {state.question}")
    
    # প্রাপ্ত ক্যাটাগরি স্টেটে সেভ করে ফেরত পাঠানো হচ্ছে
    state.category = res.category
    return state

In [8]:
# #route_function: কন্ডিশনাল এজ ডিসিশন মেকার (এটি কোনো সাধারণ নোড নয়, এটি দিকনির্দেশক)
def route(state: FlowState) -> Literal['coding', 'google_search', 'weather']:
    # স্টেটের ক্যাটাগরি পড়ে সেই অনুযায়ী পথ নির্ধারণ করবে
    return state.category

In [9]:
def weather_node(state: FlowState) -> FlowState:
    print("Weather Node")
    # এখানে ওয়েদার এপিআই বা লজিক থাকবে
    state.answer = "Weather data fetched successfully."
    return state

In [10]:
def google_search_node(state: FlowState) -> FlowState:
    res=google_agent.invoke({"messages":[
        {"role":"user","content":state.question}
    ]})
    state.answer=res["messages"][-1].content
    return state

In [11]:
def coding_node(state: FlowState) -> FlowState:
    print("Coding Node")
    # এখানে কোড জেনারেশন লজিক থাকবে
    state.answer = "Coding solution generated."
    return state

In [12]:
graph = StateGraph(FlowState)

# #AddNodes: গ্রাফের মধ্যে নোডগুলো রেজিস্টার করা
graph.add_node("check_question_category", check_question_category)
graph.add_node("weather_node", weather_node)
graph.add_node("google_search_node", google_search_node)
graph.add_node("coding_node", coding_node)

In [13]:
graph.add_edge(START, "check_question_category")
graph.add_conditional_edges(
    "check_question_category",
    route,
    {
        "coding": "coding_node",
        "google_search": "google_search_node",
        "weather": "weather_node"
    }
)

# প্রতিটি নোডের কাজ শেষ হলে গ্রাফ সমাপ্ত (END) হবে
graph.add_edge("weather_node", END)
graph.add_edge("google_search_node", END)
graph.add_edge("coding_node", END)



In [14]:
# গ্রাফ কম্পাইল
app = graph.compile()

# ইনভোক
response = app.invoke({"question": "what is location amin bazar ?"})
print(response)

{'question': 'what is location amin bazar ?', 'category': 'google_search', 'answer': 'Here\'s a comprehensive overview of **Amin Bazar**:\n\n## Location of Amin Bazar\n\n**Amin Bazar** is a union and quarter located in **Savar Upazila**, **Dhaka Division**, **Bangladesh**. It sits on the **outskirts of Dhaka city**, along the **Dhaka-Aricha highway**.\n\n### Key Geographic Details:\n- **Coordinates:** 23°47\'04"N (latitude), 90°19\'48"E (longitude)\n- **Zip Code:** 1348\n- **Position:** Just northwest of Dhaka, along the Dhaka-Aricha highway corridor\n- It is in the Savar area, about 20-25 km from central Dhaka\n\n### What Amin Bazar Is Known For:\n\n1. **Aminbazar Landfill** – The most significant landmark. This is a major municipal solid waste landfill established in **2006-2007** on approximately **52 acres** of land. It receives around **3,500 tonnes of solid waste daily** from Dhaka. However, it has also been a source of significant **environmental degradation** and **health risks